In [ ]:
import importlib.util
import json
import sys
from pathlib import Path
from typing import cast

import torch
from ignite.handlers import Checkpoint

import utils_kaggle as U

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CHECKPOINT_PATH = Path("/kaggle/input/notebooks/mustafamuhaimin/3mogenpretrain/checkpoints/pretrain")
# pretrained_checkpoint = U._find_latest_checkpoint(CHECKPOINT_PATH, "pretrain_best_val")

ckpt = U._torch_load_with_compat(
    Path("../checkpoints/finetune_latest_20260625_012629_6000.pt"), map_location=device, weights_only=False
)
ckpt_cfg = ckpt["config"]

NameError: name '__file__' is not defined

In [2]:
%pip install cattrs
from cattrs import Converter

converter = Converter()


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [ ]:
config = converter.structure(ckpt["config"], U.Config)

config.device = device
config.batch_size = 2

# print(json.dumps(config.to_dict(), indent=2))
print(type(config.encoder_config))

ema_encoder = U.MotionHistoryEncoder(config)
decoder = U.LatentDecoder(config)

Checkpoint.load_objects(
    to_load={"encoder": ema_encoder, "decoder": decoder},
    checkpoint=ckpt,
)

config.dataset_path = Path("../tests/dataset/humanml3d-subset-mini")

val_dataloader, normalizer = U.create_dataloader(config, split="val")

<class 'utils_kaggle.MotionHistoryEncoderConfig'>


100%|██████████| 8/8 [00:00<00:00, 44.94it/s]


Loading text embedding cache from ..\tests\dataset\humanml3d-subset-mini\text_embeddings_cache.pt...
All text embeddings are cached.


In [ ]:
batch = next(iter(val_dataloader))
_motion = batch["motion"][:, :80].to(config.device)
_joints = batch["joints"][:, :80].to(config.device)
_caption = batch["captions"]
_text = batch["text_clip"].to(config.device)

random_sample = torch.randint(0, _motion.shape[0], (1,))

text_clip = _text[random_sample, -1, :]
_caption[random_sample]

e:\FahadProject\BUET Lab\3-2\CSE330\ML-Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
motion = _motion[random_sample]
print(motion.shape)
with torch.inference_mode():
    # Perform reconstruction or other operations
    prev_pos = _joints[random_sample][:, :-1]
    prev_poss = torch.cat([torch.zeros_like(prev_pos[:, :1]), prev_pos], dim=1).flatten(
        0, 1
    )  # Add zero for the first frame
    prev_frame = motion[:, :1]
    prev_frames = torch.cat([prev_frame, motion[:, :-1]], dim=1).flatten(0, 1)  # (1*seq_len, motion_dim)

    latents = ema_encoder(motion, text_clip).flatten(0, 1)  # (1*seq_len, latent_dim)
    new_pos, rel_shift, _ = decoder.decode(
        latents, prev_pos=prev_poss, prev_frame=prev_frames, normalizer=normalizer
    )  # (1*seq_len, 22, 3), (1*seq_len, 22, 3), (1*seq_len, motion_dim)


joints = _joints[random_sample].cpu().numpy()[0]
positions = new_pos.numpy()

U.compare_motions(positions, joints)